# MiniLM on Colab

Runs the scripts in this repo on a free T4. Everything is saved to Drive so a
disconnect does not cost you the tokenizer, the encoded data or a checkpoint.

Runtime -> Change runtime type -> T4 GPU before you start.

In [ ]:
!git clone https://github.com/AiConsultant-SiriusAI/OwnSLM.git
%cd OwnSLM
!pip install -q -r requirements-dev.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/minilm'
!mkdir -p {DATA} checkpoints

## 1. Data

In [ ]:
!wget -nc -P {DATA} https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt
!wget -nc -P {DATA} https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-valid.txt

## 2. Tokenizer (slow: 2 CPU cores)

Drop `--max-bytes` to use the whole file. 550000000 is the first quarter.

In [ ]:
!python -m scripts.train_tokenizer --input {DATA}/TinyStoriesV2-GPT4-train.txt \n    --out-dir {DATA} --vocab-size 10000 --max-bytes 550000000

## 3. Encode both splits to uint16

In [ ]:
!python -m scripts.encode_data --data-dir {DATA}

## 4. Sanity checks before the real run

Single batch: the loss must fall close to zero. Then stop and resume.

In [ ]:
!python train.py --train-data {DATA}/train.npy --val-data {DATA}/valid.npy \n    --steps 300 --single-batch --eval-every 100 --ckpt-every 0 \n    --lr 1e-3 --amp --out checkpoints/overfit

## 5. Learning-rate pilots (5 x 300 steps)

In [ ]:
!python -m scripts.pilot_lr --steps 300 -- \n    --train-data {DATA}/train.npy --val-data {DATA}/valid.npy --amp

In [ ]:
!python -m scripts.plots checkpoints/pilot_*.csv --out pilots.png
from IPython.display import Image; Image('pilots.png')

## 6. The main run: 5000 steps x batch 32 x context 256 = 40.96M tokens

In [ ]:
!python train.py --train-data {DATA}/train.npy --val-data {DATA}/valid.npy \n    --steps 5000 --batch-size 32 --context-length 256 --lr <BEST_LR> \n    --amp --ckpt-every 500 --out {DATA}/main

In [ ]:
!python -m scripts.plots {DATA}/main.csv --out loss_steps.png
!python -m scripts.plots {DATA}/main.csv --x seconds --out loss_time.png

## 7. Release and generate

In [ ]:
!python -m scripts.make_release --checkpoint {DATA}/main.pt --data-dir {DATA} --out release
!python inference.py --model-dir release --prompt "Once upon a time" \n    --max-new-tokens 256 --temperature 0.8 --top-p 0.9 --seed 42 --device cpu